# Calculate the metrics from the csv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from evaluation_utils import bootstrap_from_csv

# --- CALCULATION ---
# Updated to match the folder structure (with "evaluation/" prefix and without "_v1")
models_to_test = {
    "DINOv2": "evaluation/csvs/dinov2_open/open_dist_matrix.csv",
    "SwinV2": "evaluation/csvs/swin_open/open_dist_matrix.csv",
    "ViT": "evaluation/csvs/vit_open/open_dist_matrix.csv"
}

# Dictionary to store all computed metrics
all_open_results = {}

for name, path in models_to_test.items():
    print(f"\n{'='*30}\nProcessing {name}\n{'='*30}")
    # Increased 'm' to 100 for statistically sound confidence intervals
    all_open_results[name] = bootstrap_from_csv(path, m=100, mode="open")

In [ ]:
#-------------------------------------------------------------
# --- Plotting Open-World Curves (DIR vs FAR) ---
# -------------------------------------------------------------

plt.figure(figsize=(10, 6))

for name, res in all_open_results.items():
    fars = res["mean_fars"]
    dirs = res["mean_dirs"]
    
    # Plot the mean DIR@FAR curve for each model
    plt.plot(fars, dirs, linewidth=2, label=f"{name}")

# Format the plot
plt.title("DIR vs FAR - Open World Evaluation", fontsize=14, fontweight='bold')
plt.xlabel("False Accept Rate (FAR)", fontsize=12)
plt.ylabel("Detection and Identification Rate (DIR)", fontsize=12)

# Zoom in on the lower FAR region (0% to 20%) since that is the critical area for open-set
plt.xlim(0, 0.20) 
plt.ylim(0, 1.05)

# Format the X-axis as percentages
ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()


# -------------------------------------------------------------
# --- Print Final Metrics with Confidence Intervals ---
# -------------------------------------------------------------
print("\n" + "="*50)
print("FINAL EVALUATION METRICS (Open-World)")
print("="*50)

for name, res in all_open_results.items():
    print(f"{name}:")
    
    fars = res["mean_fars"]
    
    # Iterate through the targets to extract the intervals directly from the arrays
    for target in [0.01, 0.05, 0.10]:
        # Find the index in the arrays closest to the target FAR
        idx = np.argmin(np.abs(fars - target))
        
        dir_mean = res["mean_dirs"][idx]
        dir_lower = res["lower_dirs"][idx]
        dir_upper = res["upper_dirs"][idx]
        
        print(f"  DIR @ FAR {target:.0%}: {dir_mean:.2%} [{dir_lower:.2%} - {dir_upper:.2%}]")
        
    print("-" * 50)

# Display the plot
plt.show()